In [1]:
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
)
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [3]:
# Reusable preprocessing import
from src.features.preprocessing import build_preprocessor

In [4]:
#Dataset + split load
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "telco_churn_clean.csv"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "split_manifest.csv"
)

df = pd.read_csv(DATA_PATH)
manifest = pd.read_csv(MANIFEST_PATH)

print("Dataset:", df.shape)
print("Manifest:", manifest.shape)

Dataset: (7043, 21)
Manifest: (7043, 3)


In [5]:
#Training data recreate
data = df.merge(
    manifest[["customerID", "split"]],
    on="customerID",
    how="left",
    validate="one_to_one"
)

train_data = data[
    data["split"] == "train"
].copy()

In [7]:
X_train = train_data.drop(
    columns=[
        "customerID",
        "Churn",
        "split",
    ]
)

y_train = train_data["Churn"].map({
    "No": 0,
    "Yes": 1,
})

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nClass counts:")
print(y_train.value_counts())

X_train: (5634, 19)
y_train: (5634,)

Class counts:
Churn
0    4139
1    1495
Name: count, dtype: int64


In [8]:
# Class imbalance ratio calculate
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = (
    negative_count / positive_count
)

print("Negative:", negative_count)
print("Positive:", positive_count)
print(
    "scale_pos_weight:",
    round(scale_pos_weight, 4)
)

Negative: 4139
Positive: 1495
scale_pos_weight: 2.7686


In [9]:
# Cross-validation strategy
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [10]:
# add 2 usefull metric
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

In [12]:
# define  6 model
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42,
    ),

    "Logistic Regression Balanced": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        n_jobs=1,
    ),

    "Random Forest Balanced": RandomForestClassifier(
        n_estimators=250,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    ),

    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        n_estimators=250,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1,
    ),

    "XGBoost Weighted": XGBClassifier(
        objective="binary:logistic",
        n_estimators=250,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1,
    ),
}

#### all model evaluate code

In [17]:
all_results = []

for model_name, model in models.items():
    print(f"Running: {model_name}")

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                build_preprocessor()
            ),
            (
                "classifier",
                model
            ),
        ]
    )

    start_time = time.time()
    scores = cross_validate(
        estimator= pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1,
    )

    elapsed_time = time.time() - start_time
    result = {
        "Model": model_name,

        "Accuracy":
            scores["test_accuracy"].mean(),

        "Balanced Accuracy":
            scores["test_balanced_accuracy"].mean(),

        "Precision":
            scores["test_precision"].mean(),

        "Recall":
            scores["test_recall"].mean(),

        "F1":
            scores["test_f1"].mean(),

        "ROC-AUC":
            scores["test_roc_auc"].mean(),

        "Average Precision":
            scores[
                "test_average_precision"
            ].mean(),

        "CV Time (sec)":
            elapsed_time,
    }

    all_results.append(result)
    print("Done.\n")

Running: Logistic Regression
Done.

Running: Logistic Regression Balanced
Done.

Running: Random Forest
Done.

Running: Random Forest Balanced
Done.

Running: XGBoost
Done.

Running: XGBoost Weighted
Done.



In [18]:
#Comparison table
comparison = pd.DataFrame(
    all_results
)

comparison = comparison.sort_values(
    by="Average Precision",
    ascending=False
).reset_index(drop=True)

comparison.round(4)

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,Average Precision,CV Time (sec)
0,XGBoost Weighted,0.7595,0.7637,0.5325,0.7726,0.6304,0.8438,0.6658,2.2758
1,XGBoost,0.8032,0.7171,0.6601,0.5338,0.5901,0.8454,0.6657,3.4647
2,Logistic Regression,0.8053,0.7237,0.6602,0.5498,0.5995,0.8458,0.6611,4.5723
3,Logistic Regression Balanced,0.7501,0.7665,0.5189,0.8013,0.6299,0.8458,0.6594,4.7357
4,Random Forest,0.7898,0.6959,0.6331,0.4957,0.5556,0.8240,0.6190,4.4451
5,Random Forest Balanced,0.7758,0.7350,0.5680,0.6482,0.6054,0.8262,0.6146,4.4552


#### Percentage-friendly table

In [19]:
display_columns = [
    "Model",
    "Accuracy",
    "Balanced Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "Average Precision",
]

comparison_display = comparison[
    display_columns
].copy()

for column in display_columns[1:]:
    comparison_display[column] = (
        comparison_display[column] * 100
    ).round(2)

comparison_display

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,Average Precision
0,XGBoost Weighted,75.95,76.37,53.25,77.26,63.04,84.38,66.58
1,XGBoost,80.32,71.71,66.01,53.38,59.01,84.54,66.57
2,Logistic Regression,80.53,72.37,66.02,54.98,59.95,84.58,66.11
3,Logistic Regression Balanced,75.01,76.65,51.89,80.13,62.99,84.58,65.94
4,Random Forest,78.98,69.59,63.31,49.57,55.56,82.40,61.90
5,Random Forest Balanced,77.58,73.50,56.80,64.82,60.54,82.62,61.46


In [20]:
#Recall ranking
comparison[
    [
        "Model",
        "Precision",
        "Recall",
        "F1",
        "Average Precision",
    ]
].sort_values(
    by="Recall",
    ascending=False
).round(4)

,Model,Precision,Recall,F1,Average Precision
3,Logistic Regression Balanced,0.5189,0.8013,0.6299,0.6594
0,XGBoost Weighted,0.5325,0.7726,0.6304,0.6658
5,Random Forest Balanced,0.5680,0.6482,0.6054,0.6146
2,Logistic Regression,0.6602,0.5498,0.5995,0.6611
1,XGBoost,0.6601,0.5338,0.5901,0.6657
4,Random Forest,0.6331,0.4957,0.5556,0.6190


#### check class balancing is true improvement ??

In [21]:
lr_comparison = comparison[
    comparison["Model"].isin([
        "Logistic Regression",
        "Logistic Regression Balanced",
    ])
][
    [
        "Model",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Average Precision",
    ]
]

lr_comparison.round(4)

,Model,Precision,Recall,F1,ROC-AUC,Average Precision
2,Logistic Regression,0.6602,0.5498,0.5995,0.8458,0.6611
3,Logistic Regression Balanced,0.5189,0.8013,0.6299,0.8458,0.6594


In [22]:
# Random Forest:
rf_comparison = comparison[
    comparison["Model"].isin([
        "Random Forest",
        "Random Forest Balanced",
    ])
][
    [
        "Model",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Average Precision",
    ]
]

rf_comparison.round(4)

,Model,Precision,Recall,F1,ROC-AUC,Average Precision
4,Random Forest,0.6331,0.4957,0.5556,0.8240,0.6190
5,Random Forest Balanced,0.5680,0.6482,0.6054,0.8262,0.6146


In [23]:
# XGBoost
xgb_comparison = comparison[
    comparison["Model"].isin([
        "XGBoost",
        "XGBoost Weighted",
    ])
][
    [
        "Model",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Average Precision",
    ]
]

xgb_comparison.round(4)

,Model,Precision,Recall,F1,ROC-AUC,Average Precision
0,XGBoost Weighted,0.5325,0.7726,0.6304,0.8438,0.6658
1,XGBoost,0.6601,0.5338,0.5901,0.8454,0.6657


### Best models print

In [24]:
best_ap_model = comparison.loc[
    comparison["Average Precision"].idxmax(),
    "Model"
]

best_f1_model = comparison.loc[
    comparison["F1"].idxmax(),
    "Model"
]

best_recall_model = comparison.loc[
    comparison["Recall"].idxmax(),
    "Model"
]

best_auc_model = comparison.loc[
    comparison["ROC-AUC"].idxmax(),
    "Model"
]

print(
    "Best Average Precision:",
    best_ap_model
)

print(
    "Best F1:",
    best_f1_model
)

print(
    "Best Recall:",
    best_recall_model
)

print(
    "Best ROC-AUC:",
    best_auc_model
)

Best Average Precision: XGBoost Weighted
Best F1: XGBoost Weighted
Best Recall: Logistic Regression Balanced
Best ROC-AUC: Logistic Regression Balanced


#### Baseline-with improvement calculate

In [25]:
baseline_row = comparison[
    comparison["Model"]
    == "Logistic Regression"
].iloc[0]

comparison["F1 vs Baseline"] = (
    comparison["F1"]
    - baseline_row["F1"]
)

comparison["Recall vs Baseline"] = (
    comparison["Recall"]
    - baseline_row["Recall"]
)

comparison[
    [
        "Model",
        "F1",
        "F1 vs Baseline",
        "Recall",
        "Recall vs Baseline",
    ]
].round(4)

,Model,F1,F1 vs Baseline,Recall,Recall vs Baseline
0,XGBoost Weighted,0.6304,0.0308,0.7726,0.2227
1,XGBoost,0.5901,-0.0094,0.5338,-0.0161
2,Logistic Regression,0.5995,0.0000,0.5498,0.0000
3,Logistic Regression Balanced,0.6299,0.0304,0.8013,0.2515
4,Random Forest,0.5556,-0.0439,0.4957,-0.0542
5,Random Forest Balanced,0.6054,0.0058,0.6482,0.0983


In [26]:
#save results

OUTPUT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "model_comparison_cv.csv"
)

comparison.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "Saved:",
    OUTPUT_PATH
)

Saved: c:\AI_Projects\customer-churn-ml-system\reports\model_comparison_cv.csv
